# Metacritic Game Site Web Scrape - Full Pipeline Write-Up

After exploring the structures of Metacritic's pages and where to find the right classes for the information we seek, We'll construct the full pipeline here.  

**Broad Work Flow**:
1. Prepare information for later lookup
    - game title
    - platforms
    - all necessary urls (use `data/mappings/platform_slugs.json`)
2. Identify which game x platform row has missing values in critic as well as user data
3. Conduct scrapes separately for
    - critic scores and review counts
    - user scores and review counts
    - game stats from backend urls
4. Aggregate all new scraped data into one df
5. Merge acquired data sensibly with original data set
6. Profit!

## Setup

In [32]:
# +++ Import all necessary modules +++

import json
import random
import time
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

from core.config import DATA_FORMATTED_PATH, MAPPINGS_DIR


In [3]:
# +++ GLOBAL SETTINGS +++

# Percentage of unique game titles to choose as batch
BATCH_PCT = 0.005


## Load data and define data sample to work with

In [ ]:
# load the data
df = pd.read_csv(DATA_FORMATTED_PATH)

unique_titles = df['title'].drop_duplicates()
n_games_total = df['title'].nunique()

# get batch size and corresponding game titles
batch_size = round(n_games_total * BATCH_PCT)
batch_titles = unique_titles.iloc[:batch_size]

# extract batch data
df_batch = df[df['title'].isin(batch_titles)].copy()

print(f"Games in full dataset:  {n_games_total}")
print(f"Games selected:         {len(batch_titles)}")
print(f"Game × platform rows:   {len(df_batch)}")

# ---
# Load the platform-slug mappings
with open(MAPPINGS_DIR / 'platform_slugs.json', 'r', encoding = 'utf-8') as f:
    platform_mapping = json.load(f)

Games in full dataset:  13429
Games selected:         67
Game × platform rows:   178


In [55]:
mask = (df['critic_total_count'] < 10) & (df['metascore'].isna())

df_lowcount = df.loc[mask]

df_lowcount[df_lowcount['critic_total_count'] == 6]

,title,genre,summary,publisherName,publisherUrl,url,original_platform,ReleaseDate,platform,metascore,...,userscore,user_positive_count,user_negative_count,user_mixed_count,user_total_count,critic_sourceVar,critic_review_url,userscore_sourceVar,user_review_sourceVar,critic_observation_count
21212,Disney Dreamlight Valley,Virtual Life,Disney Dreamlight Valley is a hybrid between a...,Gameloft,https://www.metacritic.com/company/gameloft/,https://www.metacritic.com/game/disney-dreamli...,PC,2023-12-05,PlayStation 5,NaN,...,NaN,NaN,NaN,NaN,NaN,platformReviews/0,https://www.metacritic.com/game/disney-dreamli...,NaN,NaN,1
26007,GetsuFumaDen: Undying Moon,2D Platformer,Getsu Fuma Descends It has been 1000 years sin...,Konami,https://www.metacritic.com/company/konami/,https://www.metacritic.com/game/getsufumaden-u...,Nintendo Switch,2022-02-09,PC,NaN,...,NaN,NaN,NaN,NaN,NaN,platformReviews/1,https://www.metacritic.com/game/getsufumaden-u...,NaN,NaN,1


In [36]:
# Create container to gather scraped data in
scraped_data = {}

for _, row in df_batch.iterrows():
    key = (row["title"], row["platform"])

    scraped_data[key] = {
        "metascore": None,
        "critic_positive_count": None,
        'critic_positive_pc': None,
        "critic_negative_count": None,
        'critic_negative_pc': None,
        "critic_mixed_count": None,
        'critic_mixed_pc': None,
        "userscore": None,
        "user_positive_count": None,
        "user_positive_pc": None,
        "user_negative_count": None,
        "user_negative_pc": None,
        "user_mixed_count": None,
        "user_mixed_pc": None,
    }

## 1. Prepare Look-Up Information

In [ ]:
# gather title, available platforms, and corresponding critic-review and user-review urls
# for each game in the batch

game_lookup = {}

for title, game_df in df_batch.groupby("title", sort=False):

    main_url = game_df["url"].iloc[0] # game main page

    game_lookup[title] = {
        "main_url": main_url,
        "platforms": {}
    }

    for platform in game_df["platform"].dropna().unique():

        slug = platform_mapping[platform]

        game_lookup[title]["platforms"][platform] = {
            "slug": slug,
            "critic_url": f"{main_url}/critic-reviews/?platform={slug}",
            "user_url": f"{main_url}/user-reviews/?platform={slug}"
        }

## 2. Identify game entries with missing information

- Data gathering will be conducted only for the game x platform combinations that actually require it.
- The approach requires separate row selections for critic and user content scrapes

In [ ]:
# Define variables to be considered for the selection
critic_cols = ['metascore',
               'critic_positive_count',
               'critic_negative_count',
               'critic_mixed_count']
user_cols = ['userscore',
             'user_positive_count',
             'user_negative_count',
             'user_mixed_count']

# create selection masks
critic_mask = df_batch[critic_cols].isna().any(axis = 1)
user_mask = df_batch[user_cols].isna().any(axis = 1)

# subset batch data accordingly
critic_scrape_batch = df_batch.loc[critic_mask].copy()
user_scrape_batch = df_batch.loc[user_mask].copy()

## 3. Gather missing data on Critic Reviews

In [46]:
# +++ Helper Functions +++

def extract_review_count(soup, review_type, game, platform):
    """
    Extract the review count and percentage for a given review category.

    PARAMETERS:
        soup : BeautifulSoup
            Parsed HTML of a Metacritic review page.
        review_type : str
            Review category to extract, e.g. "positive", "mixed", or "negative".

    RETURNS:
        count : str
            Review count as displayed on the page.
        pc : str
            Percentage of reviews belonging to the specified category.
    """
    label = soup.find(
        "div",
        string = lambda text: (
            text
            and text.strip().lower() == review_type.strip().lower()
        ),
    )

    # catch if there's not even a single review
    if label is None:
        print(f'  No reviews for {game} ({platform})')
        return None, None

    count_span = label.find_next('span')
    pc_span = count_span.find_next('span')

    count = count_span.get_text(strip=True)
    pc = pc_span.get_text(strip=True)

    return count, pc

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0'}

for row in critic_scrape_batch.itertuples(index = False):

    game = row.title
    pf = row.platform

    print(f'Working on {game} ...')
    pf_url = game_lookup[game]['platforms'][pf]['critic_url']

    resp = requests.get(pf_url, headers = headers)
    resp.raise_for_status()

    lag = resp.elapsed.total_seconds()

    html = resp.text
    soup = BeautifulSoup(html, 'html.parser')

    score_div = soup.find_all(class_ = "score-card-left__score-number")

    # catch case that there is no meta score for a platform at all
    if score_div:
        score_str = score_div[0].find('span').get_text(strip = True)
    else:
        score_str = None

    # extract review counts for positive, negative, and mixed reviews
    pos_count, pos_pc = extract_review_count(soup, 'positive', game, pf)
    # Catch case of no reviews present at all.
    if pos_count is not None:
        neg_count, neg_pc = extract_review_count(soup, 'negative', game, pf)
        mix_count, mix_pc = extract_review_count(soup, 'mixed', game, pf)
    else:
        neg_count, neg_pc = None, None
        mix_count, mix_pc = None, None

    # gather up the data
    key = (game, pf)
    scraped_data[key].update({
        'metascore': score_str,
        'critic_positive_count': pos_count,
        'critic_positive_pc': pos_pc,
        'critic_negative_count': neg_count,
        'critic_negative_pc': neg_pc,
        'critic_mixed_count': mix_count,
        'critic_mixed_pc': mix_pc,
    })

    time.sleep(random.uniform(1,2) * lag)